# Snowflake CLI で customers.csv を @WORKSHOP_STAGE にアップロードする手順

## 1. Snowflake CLI のインストール

**方法A: pip でインストール**
```bash
pip install snowflake-cli
```

**方法B: MSI インストーラー (Windows)**

[Snowflake CLI インストーラー](https://sfc-repo.snowflakecomputing.com/snowflake-cli/index.html) からMSI版をダウンロードして実行します。

## 2. 接続設定

### アカウント情報の確認
Snowsight 左下のアカウントメニューから **「アカウントの詳細を表示する」** を選択すると、アカウント識別子やリージョン情報を確認できます。
ここで **「Config File」** タブを開くと、接続設定に必要な情報がまとめて表示されます。

### 接続の追加
```bash
snow connection add
```

以下のように入力します：

| プロンプト | 入力例 |
|---|---|
| Name | `my_connection` |
| Account | Config File に表示されたアカウント識別子 |
| User | `ADMIN1` |
| Password | ご自身のパスワード |
| Role | `MU` |
| Warehouse | `MUWH1` |
| Database | `SECURITIES_WORKSHOP` |
| Schema | `RAW_DATA` |
| Authentication method | なにも入力せず Enterキー |

> **ヒント:** アカウント識別子は `<orgname>-<account_name>` の形式です（例: `MYORG-wc41166`）。Config File からそのままコピーできます。

### 接続の使い方

各コマンド実行時に接続を指定するには `--connection` フラグを使います：
```bash
snow sql -q "SELECT CURRENT_USER()" --connection <接続名>
```

`default_connection_name` に設定した接続はフラグなしで自動的に使われます：
```bash
snow connection set-default <接続名>
```

## 3. ファイルをステージにアップロード

本ハンズオンでは、CSVファイルを `C:\tempdata` フォルダに配置している前提とします。

Windows のコマンドプロンプト（または PowerShell）で以下を実行します：

```bash
snow stage copy C:\tempdata\customers.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
```

> **注意: PowerShell での `@` について**
> PowerShell では `@` が特殊文字（スプラッティング演算子）として解釈されるため、ステージ名は必ずダブルクォートで囲んでください。

> **注意: Windows でのパス指定について**
> - バックスラッシュ (`\`) をそのまま使用できます
> - パスにスペースが含まれる場合はダブルクォートで囲みます：
>   ```bash
>   snow stage copy "C:\My Folder\customers.csv" "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
>   ```
> - 複数ファイルをまとめてアップロードする場合：
>   ```bash
>   snow stage copy C:\tempdata\*.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
>   ```

### 圧縮してアップロードする場合
`snow stage copy` はデフォルトでは **圧縮せずに** アップロードします。
圧縮する場合は、`--auto-compress` オプションを指定します：

```bash
snow stage copy C:\tempdata\customers.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE" --auto-compress
```

> **使い分け:**
> - 圧縮なし（デフォルト）→ ステージ上でそのままプレビュー可能。シンプルな運用向け
> - 圧縮あり → 転送が速く、ストレージ節約。`COPY INTO` 時に自動解凍される

## 4. アップロード確認
```bash
snow stage list-files "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
```

### ディレクトリテーブルの有効化
ステージ上のファイルを SQL からテーブルのように検索したい場合は、**ディレクトリテーブル**を有効化します：

```sql
ALTER STAGE SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE SET DIRECTORY = (ENABLE = TRUE);
```

有効化後、メタデータを手動リフレッシュします：

```sql
ALTER STAGE SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE REFRESH;
```

これにより、ステージ上のファイル一覧を SQL で確認できます：

```sql
SELECT * FROM DIRECTORY(@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE);
```

> **補足:** ディレクトリテーブルはファイルのパス・サイズ・最終更新日時などのメタデータを保持します。ファイルを追加・削除した後は `ALTER STAGE ... REFRESH` で メタデータを更新してください。

> **補足:** `snow stage copy` は内部的に `PUT` コマンドを実行します。

# Snowsight GUI で holdings.csv を @WORKSHOP_STAGE にアップロードする手順

## 1. ステージ画面を開く
1. Snowsight 左メニューから **カタログ** → **データベースエクスプローラー** を選択
2. **SECURITIES_WORKSHOP** → **RAW_DATA** → **ステージ** と展開
3. **WORKSHOP_STAGE** をクリック

## 2. ファイルをアップロード
1. 画面右上の **「+ ファイル」** ボタンをクリック
2. ファイル選択ダイアログが開くので、`C:\tempdata\holdings.csv` を選択
3. **「アップロード」** をクリック

## 3. アップロード確認
ステージ画面のファイル一覧に `holdings.csv` が表示されていれば完了です。

> **補足:** GUI からのアップロードは **50MB** までのファイルに対応しています。それ以上のファイルは Snowflake CLI や SnowSQL の `PUT` コマンドを使用してください。

# 4. 残りのCSVファイルをアップロード

以下の3ファイルも同様に `@WORKSHOP_STAGE` にアップロードします。

| ファイル名 | 内容 |
|---|---|
| `market_prices.csv` | 市場価格データ |
| `target_allocation.csv` | 目標配分データ |
| `daily_trades.csv` | 日次取引データ |

### Snowflake CLI の場合
```bash
snow stage copy C:\tempdata\market_prices.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
snow stage copy C:\tempdata\target_allocation.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
snow stage copy C:\tempdata\daily_trades.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
```

または、ワイルドカードでまとめてアップロード：
```bash
snow stage copy C:\tempdata\*.csv "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
```

### Snowsight GUI の場合
1. **WORKSHOP_STAGE** のステージ画面を開く
2. **「+ ファイル」** をクリック
3. `Ctrl` キーを押しながら3ファイルをまとめて選択
4. **「アップロード」** をクリック

### アップロード確認
```bash
snow stage list-files "@SECURITIES_WORKSHOP.RAW_DATA.WORKSHOP_STAGE"
```

全5ファイルが表示されていれば完了です：
- `customers.csv`
- `holdings.csv`
- `market_prices.csv`
- `target_allocation.csv`
- `daily_trades.csv`